<a href="https://colab.research.google.com/github/smkalle/arxiv_impl/blob/main/plan1_lstm_prophet_buoy_scheduler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ Plan 1 — Hierarchical Energy-Aware Camera Scheduler
## LSTM (24 h tactical) × Prophet (7-day strategic) — *no TimeGPT*

**Problem.** A solar-powered camera buoy at an IMTA site (Gulf Coast, Alabama) has:

| Parameter | Value |
|---|---|
| Solar panel | 100 W rated |
| Battery | 1200 Wh |
| Camera draw | 33.7 W |
| Typical harvest | ≈ 230 Wh/day |
| Safety floor | 20 % SoC (240 Wh) — **hard constraint** |

The camera cannot run 24/7 (33.7 W × 24 h = 808.8 Wh ≫ 230 Wh harvested).
**Goal:** maximize captured footage hours while *never* crossing the SoC floor.

**Architecture** — this is hierarchical model-predictive control, not "just forecasting":

```
┌──────────────────────────────────────────────────────────────┐
│  STRATEGIC LAYER (nightly)                                   │
│  Prophet 7-day daily-harvest forecast ──► Budget LP          │
│  output: daily_budget_Wh[d]  for d = 0..6                    │
└───────────────┬──────────────────────────────────────────────┘
                │ downward coupling: today's Wh budget
┌───────────────▼──────────────────────────────────────────────┐
│  TACTICAL LAYER (nightly, per-day)                           │
│  LSTM 24-h hourly-solar forecast ──► Hour Selector           │
│  output: camera_on[h]  for h = 0..23  (solar-coincident)     │
└───────────────┬──────────────────────────────────────────────┘
                │ execution against *real* weather
┌───────────────▼──────────────────────────────────────────────┐
│  UPWARD COUPLING (each morning)                              │
│  • actual end-of-day SoC re-anchors the 7-day LP             │
│  • LSTM-vs-Prophet residual bias-corrects remaining 6 days   │
└──────────────────────────────────────────────────────────────┘
```

The **shared state that wires the two models together is the battery SoC trajectory** —
Prophet decides *how much* energy each day may spend; the LSTM decides *which hours*
spend it, preferring hours where forecast harvest ≥ camera draw ("run off the panel,
not off the battery").

**Notebook map**
1. Setup, logging, constants
2. Buoy energy model
3. Synthetic Gulf Coast solar data (with a frontal-passage storm)
4. LSTM 24-h forecaster (target: ≤ 0.93 W MAE) — with pure-NumPy fallback
5. Prophet 7-day forecaster — with seasonal-naive fallback
6. Layer A: weekly budget optimizer (linear program)
7. Layer B: solar-coincidence hour selector
8. The joint receding-horizon loop (30-day closed-loop simulation)
9. Baselines & evaluation
10. Uncertainty handling & deployment notes


## 1. Setup, Logging & Reproducibility

We configure structured logging so every planning decision leaves an audit trail —
essential when you later ask *"why did the buoy conserve on Tuesday?"*

Optional heavy dependencies (`torch`, `prophet`) are detected at import time.
If missing, the notebook **still runs end-to-end** using documented fallback
forecasters, so the *control architecture* can be studied independently of the ML stack.

In [ ]:
# --- optional installs (uncomment on first run) -----------------------------
# %pip install torch --index-url https://download.pytorch.org/whl/cpu -q
# %pip install prophet -q

import logging, sys, math, warnings
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

warnings.filterwarnings("ignore")
np.random.seed(42)

# ---- structured logger ------------------------------------------------------
log = logging.getLogger("buoy")
log.setLevel(logging.INFO)
log.handlers.clear()
_h = logging.StreamHandler(sys.stdout)
_h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-5s | %(message)s", "%H:%M:%S"))
log.addHandler(_h)

# ---- optional deps ----------------------------------------------------------
try:
    import torch, torch.nn as nn
    TORCH_OK = True
    torch.manual_seed(42)
except ImportError:
    TORCH_OK = False

try:
    from prophet import Prophet
    PROPHET_OK = True
except ImportError:
    PROPHET_OK = False

plt.rcParams.update({"figure.figsize": (13, 4), "figure.dpi": 100,
                     "axes.grid": True, "grid.alpha": 0.3})

log.info(f"PyTorch available: {TORCH_OK} | Prophet available: {PROPHET_OK}")
log.info("Fallback forecasters will be used for any missing dependency — "
         "the control loop is identical either way.")

## 2. The Buoy Energy Model

The deterministic core **both layers share**. One equation drives everything:

$$SoC_{t+1} = \mathrm{clip}\Big(SoC_t + \eta_{chg}\cdot harvest_t - camera_t\cdot 33.7 - P_{hotel},\; 240,\; 1200\Big)$$

Key economics (why the problem is hard):

* Harvest ≈ 230 Wh/day, hotel load 3 W × 24 h = 72 Wh/day
* Usable surplus ≈ **(230 × 0.9) − 72 ≈ 135 Wh/day → ~4 camera-hours/day average**
* Every hour scheduled *during* sunshine is nearly free; every night hour costs full battery
* Charging stops when the battery is full → **surplus sun is wasted ("clipping")** unless
  the camera is running to absorb it. Good schedulers spend during clipping risk.

In [ ]:
@dataclass
class BuoyConfig:
    """All physical constants for the buoy energy system (single source of truth)."""
    battery_wh: float      = 1200.0   # nameplate capacity
    soc_floor_wh: float    = 240.0    # 20% hard floor (BMS cutoff proxy)
    camera_w: float        = 33.7     # camera power draw
    hotel_w: float         = 3.0      # controller + telemetry baseline
    charge_eff: float      = 0.90     # PV -> battery round-trip efficiency
    panel_w: float         = 100.0    # rated panel size (caps harvest)

    @property
    def usable_wh(self):  return self.battery_wh - self.soc_floor_wh
    @property
    def camera_wh_per_h(self): return self.camera_w

CFG = BuoyConfig()

def step_soc(soc, harvest_w, camera_on, cfg=CFG):
    """Advance SoC by one hour. Returns (new_soc, clipped_wh).

    clipped_wh = solar energy *thrown away* because the battery was full.
    Tracking it lets us score schedulers on waste, not just safety."""
    inflow  = cfg.charge_eff * min(harvest_w, cfg.panel_w)
    outflow = cfg.hotel_w + (cfg.camera_w if camera_on else 0.0)
    raw = soc + inflow - outflow
    clipped = max(0.0, raw - cfg.battery_wh)
    return min(max(raw, 0.0), cfg.battery_wh), clipped

def simulate_day(soc0, harvest_w_24, schedule_24, cfg=CFG):
    """Simulate 24 h. Returns (soc_series[25], clipped_total, floor_violated)."""
    soc, socs, clip_tot, violated = soc0, [soc0], 0.0, False
    for h in range(24):
        soc, c = step_soc(soc, harvest_w_24[h], schedule_24[h], cfg)
        clip_tot += c
        if soc < cfg.soc_floor_wh: violated = True
        socs.append(soc)
    return np.array(socs), clip_tot, violated

# ---- sanity check: the "tight economics" claim ------------------------------
daily_harvest_wh   = 230 * CFG.charge_eff
daily_hotel_wh     = CFG.hotel_w * 24
surplus            = daily_harvest_wh - daily_hotel_wh
log.info(f"Energy ledger: harvest {daily_harvest_wh:.0f} Wh − hotel {daily_hotel_wh:.0f} Wh "
         f"= {surplus:.0f} Wh surplus → {surplus/CFG.camera_w:.1f} camera-hours/day sustainable")

## 3. Synthetic Gulf Coast Solar Data

We generate **120 days of hourly PV power** for ≈ 30.3° N (Mobile Bay area):

1. **Clear-sky curve** from simple solar geometry (declination + hour angle → elevation → power).
2. **Cloud regimes** via a 3-state Markov chain — `CLEAR`, `PARTLY` (Gulf afternoon
   convective pop-ups), `FRONTAL` (2–3 day storm systems). We *force* a frontal passage
   into the final evaluation month so the closed loop is tested under stress.
3. Amplitude tuned so mean daily harvest ≈ **230 Wh/day** — matching the real buoy.

The generator returns ground truth we later hide from the forecasters (train/test hygiene).

In [ ]:
MARINE_DERATE = 0.70   # haze, salt film, non-optimal tilt on a bobbing buoy —
                       # calibrates mean daily harvest to the observed ≈230 Wh

def clear_sky_power(day_of_year, hour, lat_deg=30.3, panel_w=100.0, derate=MARINE_DERATE):
    """Idealized PV power (W) from solar elevation. Simple but has correct
    seasonal + diurnal shape, which is all the scheduler needs."""
    decl = 23.45 * math.sin(math.radians(360 * (284 + day_of_year) / 365))
    ha   = 15 * (hour - 12)                                   # hour angle, deg
    lat, decl, ha = map(math.radians, (lat_deg, decl, ha))
    sin_elev = (math.sin(lat)*math.sin(decl) +
                math.cos(lat)*math.cos(decl)*math.cos(ha))
    if sin_elev <= 0: return 0.0
    # air-mass attenuation approximation, then marine derate
    return derate * panel_w * sin_elev * (0.7 ** ((1/max(sin_elev,0.05)) ** 0.678))

REGIMES = {"CLEAR": (0.98, 0.05), "PARTLY": (0.62, 0.22), "FRONTAL": (0.18, 0.10)}
TRANS   = {"CLEAR":   {"CLEAR": .70, "PARTLY": .25, "FRONTAL": .05},
           "PARTLY":  {"CLEAR": .35, "PARTLY": .50, "FRONTAL": .15},
           "FRONTAL": {"CLEAR": .15, "PARTLY": .45, "FRONTAL": .40}}

def generate_buoy_data(n_days=120, start="2025-03-01", force_frontal_at=None, seed=42):
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, periods=n_days*24, freq="h")
    regime, rows = "CLEAR", []
    regimes_by_day, factors_by_day = [], []
    for d in range(n_days):
        # Markov step (or forced storm for stress-testing)
        if force_frontal_at and d in force_frontal_at:
            regime = "FRONTAL"
        else:
            p = TRANS[regime]; regime = rng.choice(list(p), p=list(p.values()))
        regimes_by_day.append(regime)
        base_f, noise_f = REGIMES[regime]
        day_factor = np.clip(rng.normal(base_f, noise_f/2), 0.05, 1.0)
        factors_by_day.append(day_factor)
        for h in range(24):
            cs = clear_sky_power(idx[d*24+h].dayofyear, h)
            # intra-hour cloud texture: partly-cloudy is spiky, frontal is flat-low
            tex = np.clip(rng.normal(1.0, noise_f), 0.0, 1.25)
            rows.append(max(0.0, cs * day_factor * tex))
    df = pd.DataFrame({"timestamp": idx, "solar_w": rows})
    df["clearsky_w"] = [clear_sky_power(t.dayofyear, t.hour) for t in idx]
    df["hour"] = idx.hour; df["date"] = idx.date
    return df, regimes_by_day, factors_by_day

# Final 30 days are the evaluation window; force a 3-day frontal passage on eval days 12-14.
N_TRAIN_DAYS, N_EVAL_DAYS = 90, 30
STORM_DAYS = {N_TRAIN_DAYS+12, N_TRAIN_DAYS+13, N_TRAIN_DAYS+14}
df, regimes, true_factors = generate_buoy_data(N_TRAIN_DAYS + N_EVAL_DAYS, force_frontal_at=STORM_DAYS)

daily = df.groupby("date")["solar_w"].sum().rename("harvest_wh").to_frame()
daily["regime"] = regimes
daily["true_factor"] = true_factors
daily["cs_wh"] = df.groupby("date")["clearsky_w"].sum().values   # clear-sky daily energy
log.info(f"Generated {len(df)} hourly rows over {N_TRAIN_DAYS+N_EVAL_DAYS} days")
log.info(f"Mean daily harvest: {daily['harvest_wh'].mean():.0f} Wh "
         f"(target ≈230) | storm days: {sorted(STORM_DAYS)}")

In [ ]:
# ---- visual sanity check ----------------------------------------------------
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=False)

wk = df.iloc[:7*24]
ax[0].plot(wk.timestamp, wk.solar_w, lw=.8, label="actual PV power")
ax[0].plot(wk.timestamp, wk.clearsky_w, "--", lw=1, color="orange", label="clear-sky envelope")
ax[0].axhline(CFG.camera_w, color="r", ls=":", label=f"camera draw {CFG.camera_w} W")
ax[0].set(title="First week — hourly PV vs clear-sky envelope", ylabel="Watts"); ax[0].legend()

colors = {"CLEAR": "#2ca02c", "PARTLY": "#ff7f0e", "FRONTAL": "#d62728"}
ax[1].bar(range(len(daily)), daily.harvest_wh, color=[colors[r] for r in daily.regime], width=1.0)
ax[1].axhline(230, color="k", ls="--", lw=1, label="230 Wh nominal")
ax[1].axvspan(N_TRAIN_DAYS-0.5, len(daily)-0.5, alpha=0.08, color="blue", label="evaluation window")
for d in STORM_DAYS: ax[1].axvline(d, color="navy", lw=.5, alpha=.5)
ax[1].set(title="Daily harvest, colored by cloud regime (green=clear, orange=partly, red=frontal)",
          xlabel="day index", ylabel="Wh/day"); ax[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

log.info("Note the red frontal days — harvest collapses to ~40-80 Wh. "
         "A scheduler that ignores the 7-day outlook will hit the floor there.")

## 4. Tactical Forecaster — LSTM, 24-hour horizon

**Design** (mirrors the research benchmark that achieved 0.93 W MAE):

* **Input window:** last 72 h of `[solar_w, clearsky_w, sin(hour), cos(hour)]`
* **Output:** next 24 hourly power values (direct multi-step, avoids error feedback)
* **Normalization:** divide power by panel rating (100 W) → all features in ~[0, 1]
* 2-layer LSTM (64 units) → Linear(24). Small on purpose: this must run shore-side nightly.

**Fallback** (`torch` absent): *clear-sky × recent-cloudiness* — tomorrow's power =
clear-sky curve scaled by the median (actual / clearsky) ratio of the last 3 daylight days.
This is a genuinely strong baseline for solar and keeps the notebook runnable anywhere.
The rest of the pipeline is forecaster-agnostic: anything exposing
`predict_next_24h(history_df) -> np.array(24)` plugs in.

In [ ]:
FEATS = ["solar_w", "clearsky_w", "sin_h", "cos_h"]

def add_time_feats(d):
    d = d.copy()
    d["sin_h"] = np.sin(2*np.pi*d.hour/24); d["cos_h"] = np.cos(2*np.pi*d.hour/24)
    return d

def make_windows(d, in_len=72, out_len=24):
    """Sliding windows aligned to midnight boundaries → X:(n,72,4)  y:(n,24)."""
    d = add_time_feats(d)
    arr = d[FEATS].values.astype(np.float32); arr[:, :2] /= CFG.panel_w
    X, y = [], []
    for start in range(0, len(d) - in_len - out_len + 1, 24):   # daily stride
        X.append(arr[start:start+in_len])
        y.append(arr[start+in_len:start+in_len+out_len, 0])
    return np.array(X), np.array(y)

class FallbackSolarForecaster:
    """Clear-sky × recent-cloudiness ratio. No learned parameters."""
    name = "clearsky-ratio fallback"
    def fit(self, hist): return self
    def predict_next_24h(self, hist):
        h = add_time_feats(hist).iloc[-72:]
        day = (h.clearsky_w > 5)
        ratio = np.clip(np.median((h.solar_w[day] / h.clearsky_w[day])), 0.05, 1.1) if day.any() else 0.5
        last_ts = hist.timestamp.iloc[-1]
        return np.array([clear_sky_power((last_ts + pd.Timedelta(hours=k+1)).dayofyear,
                                         (last_ts + pd.Timedelta(hours=k+1)).hour) * ratio
                         for k in range(24)])

if TORCH_OK:
    class LSTMSolarForecaster(nn.Module):
        name = "LSTM(2x64)"
        def __init__(s, in_dim=4, hid=64, out_len=24):
            super().__init__()
            s.lstm = nn.LSTM(in_dim, hid, num_layers=2, batch_first=True, dropout=0.1)
            s.head = nn.Linear(hid, out_len)
        def forward(s, x): return s.head(s.lstm(x)[1][0][-1])
        def fit(s, hist, epochs=60, lr=1e-3, val_frac=0.15):
            X, y = make_windows(hist)
            n_val = max(1, int(len(X)*val_frac))
            Xt, yt = torch.tensor(X[:-n_val]), torch.tensor(y[:-n_val])
            Xv, yv = torch.tensor(X[-n_val:]), torch.tensor(y[-n_val:])
            opt, lossf = torch.optim.Adam(s.parameters(), lr=lr), nn.L1Loss()
            hist_tr, hist_va, best = [], [], (1e9, None)
            for ep in range(epochs):
                s.train(); opt.zero_grad()
                l = lossf(s(Xt), yt); l.backward(); opt.step()
                s.eval()
                with torch.no_grad(): lv = lossf(s(Xv), yv).item()
                hist_tr.append(l.item()); hist_va.append(lv)
                if lv < best[0]: best = (lv, {k: v.clone() for k, v in s.state_dict().items()})
                if ep % 10 == 0: log.info(f"  epoch {ep:3d} | train MAE {l.item()*100:.2f} W | val MAE {lv*100:.2f} W")
            s.load_state_dict(best[1]); s._curves = (hist_tr, hist_va)
            return s
        def predict_next_24h(s, hist):
            arr = add_time_feats(hist).iloc[-72:][FEATS].values.astype(np.float32)
            arr[:, :2] /= CFG.panel_w
            s.eval()
            with torch.no_grad():
                out = s(torch.tensor(arr).unsqueeze(0)).numpy().ravel()
            return np.clip(out * CFG.panel_w, 0, CFG.panel_w)

train_df = df[df.timestamp < df.timestamp.iloc[0] + pd.Timedelta(days=N_TRAIN_DAYS)]
log.info(f"Training tactical forecaster on {N_TRAIN_DAYS} days "
         f"({'LSTM' if TORCH_OK else 'FALLBACK — install torch for the real model'})")
tactical = (LSTMSolarForecaster().fit(train_df) if TORCH_OK
            else FallbackSolarForecaster().fit(train_df))

In [ ]:
# ---- evaluate the 24h forecaster on held-out eval days ----------------------
def eval_tactical(model, full_df, eval_days):
    """Walk-forward: for each eval day, predict its 24h from all prior history."""
    maes, all_pred, all_true = [], [], []
    for d in eval_days:
        cut = full_df.timestamp.iloc[0] + pd.Timedelta(days=d)
        hist = full_df[full_df.timestamp < cut]
        true = full_df[(full_df.timestamp >= cut) &
                       (full_df.timestamp < cut + pd.Timedelta(days=1))].solar_w.values
        if len(true) < 24: continue
        pred = model.predict_next_24h(hist)
        maes.append(np.mean(np.abs(pred - true)))
        all_pred.append(pred); all_true.append(true)
    return np.array(maes), np.array(all_pred), np.array(all_true)

eval_day_ids = list(range(N_TRAIN_DAYS, N_TRAIN_DAYS + N_EVAL_DAYS))
maes, preds, trues = eval_tactical(tactical, df, eval_day_ids)
log.info(f"Tactical 24h forecaster [{tactical.name}] — walk-forward over {len(maes)} days:")
log.info(f"  MAE {maes.mean():.2f} W  (benchmark target: 0.93 W with tuned LSTM on real data)")
log.info(f"  worst day MAE {maes.max():.2f} W (storm days are hardest — expected)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sample = [0, 12]  # a normal day and the first storm day
for i, s in enumerate(sample):
    ax[i].plot(trues[s], label="actual", lw=2)
    ax[i].plot(preds[s], "--", label="forecast", lw=2)
    ax[i].axhline(CFG.camera_w, color="r", ls=":", lw=1, label="camera draw")
    ax[i].set(title=f"Eval day {sample[i]} ({'storm' if eval_day_ids[s] in STORM_DAYS else 'normal'}) "
                    f"— MAE {maes[s]:.1f} W", xlabel="hour", ylabel="W")
    ax[i].legend()
plt.tight_layout(); plt.show()

if TORCH_OK and hasattr(tactical, "_curves"):
    tr, va = tactical._curves
    plt.figure(figsize=(7,3))
    plt.plot(np.array(tr)*100, label="train MAE (W)"); plt.plot(np.array(va)*100, label="val MAE (W)")
    plt.title("LSTM training curves"); plt.xlabel("epoch"); plt.legend(); plt.tight_layout(); plt.show()

## 5. Strategic Forecaster — 7-day horizon (Prophet × Weather Outlook)

**A hard-won lesson (we hit it in testing):** a purely statistical model — Prophet *or*
any trailing-window method — extrapolates history. A Gulf frontal system arriving
tomorrow is **invisible in yesterday's data**, so a stats-only strategic layer budgets
generously right into the storm and breaches the floor. This is not a model bug;
it's an information bug.

**Fix, exactly as in real operations:** decompose the forecast as

$$\hat H_d \;=\; \underbrace{E^{clearsky}_d}_{\text{deterministic (solar geometry)}} \times \underbrace{\hat f_d}_{\text{cloud factor from a weather outlook (NWS/Open-Meteo)}}$$

We simulate an **NWS-like 7-day cloud outlook**: accurate at short leads
(±5 % day 1), degrading with lead time, and regressing to climatology beyond day 3 —
matching how real NWP skill decays. Prophet (or the trailing-quantile fallback) remains
in the stack as a *bias monitor and blend prior*, but the storm signal comes from the
outlook. The conservative band `yhat_lower` = clear-sky × pessimistic cloud factor.

> In deployment: replace `simulated_weather_outlook()` with an Open-Meteo /
> NWS API call returning daily cloud-cover fractions. The interface is identical.

In [ ]:
def simulated_weather_outlook(day_id, horizon=7, seed_shift=0):
    """NWS-like cloud-factor outlook: (median, pessimistic) per lead day.
    Lead-time-dependent noise + regression to climatology beyond ~3 days."""
    rng = np.random.default_rng(10_000 + day_id + seed_shift)
    clim = 0.75                                            # long-run mean cloud factor here
    f_med, f_lo = [], []
    for L in range(horizon):
        true_f = daily.true_factor.iloc[day_id + L] if day_id + L < len(daily) else clim
        sigma  = 0.05 + 0.05*L                             # skill decays with lead
        noisy  = float(np.clip(rng.normal(true_f, sigma), 0.05, 1.05))
        w      = math.exp(-max(0, L-2)/2.5)                # blend to climatology after day 3
        med    = w*noisy + (1-w)*clim
        f_med.append(med)
        f_lo.append(max(0.05, med*(0.85 - 0.04*L)))        # widening pessimistic band
    return np.array(f_med), np.array(f_lo)

class FallbackWeeklyForecaster:
    """Stats-only trailing quantile — kept as the bias-monitor prior."""
    name = "trailing-quantile"
    def predict_next_7d(self, daily_hist):
        tail = daily_hist.harvest_wh.iloc[-14:]
        return pd.DataFrame({"yhat": [tail.median()]*7,
                             "yhat_lower": [tail.quantile(0.2)]*7})

class ProphetWeeklyForecaster:
    name = "Prophet(weekly)"
    def predict_next_7d(self, daily_hist):
        pdf = daily_hist.reset_index().rename(columns={"date": "ds", "harvest_wh": "y"})
        pdf["ds"] = pd.to_datetime(pdf["ds"])
        m = Prophet(weekly_seasonality=True, yearly_seasonality=False,
                    daily_seasonality=False, interval_width=0.8,
                    changepoint_prior_scale=0.5)
        import logging as _l; _l.getLogger("cmdstanpy").setLevel(_l.ERROR)
        m.fit(pdf)
        fut = m.make_future_dataframe(periods=7)
        return m.predict(fut).iloc[-7:][["yhat","yhat_lower"]].clip(lower=0).reset_index(drop=True)

class OutlookStrategic:
    """Production pattern: clear-sky × weather-outlook, with the statistical model
    as a sanity prior (blended 20%). This is what the hierarchical policy uses."""
    name = "clearsky × NWS-outlook (+stats prior)"
    def __init__(self, stats_model): self.stats = stats_model
    def predict_next_7d(self, daily_hist):
        d0 = len(daily_hist)                               # current day index
        f_med, f_lo = simulated_weather_outlook(d0)
        cs = np.array([daily.cs_wh.iloc[min(d0+L, len(daily)-1)] for L in range(7)])
        stats_fc = self.stats.predict_next_7d(daily_hist)
        yhat = 0.8*(cs*f_med) + 0.2*stats_fc.yhat.values
        ylo  = np.minimum(cs*f_lo, stats_fc.yhat_lower.values*1.5)
        return pd.DataFrame({"yhat": yhat, "yhat_lower": ylo})

stats_prior = ProphetWeeklyForecaster() if PROPHET_OK else FallbackWeeklyForecaster()
strategic = OutlookStrategic(stats_prior)
log.info(f"Strategic forecaster: {strategic.name} (stats prior: {stats_prior.name})")

# quick visual: forecast the first eval week from training data
fc7 = strategic.predict_next_7d(daily.iloc[:N_TRAIN_DAYS])
actual7 = daily.harvest_wh.iloc[N_TRAIN_DAYS:N_TRAIN_DAYS+7].values
plt.figure(figsize=(9,3.5))
plt.plot(range(7), actual7, "o-", label="actual harvest")
plt.plot(range(7), fc7.yhat, "s--", label="yhat (outlook-blended)")
plt.fill_between(range(7), fc7.yhat_lower, fc7.yhat, alpha=.2, label="conservative band")
plt.title("7-day strategic forecast vs actual (first eval week)")
plt.xlabel("days ahead"); plt.ylabel("Wh/day"); plt.legend(); plt.tight_layout(); plt.show()
log.info(f"7-day MAE {np.mean(np.abs(fc7.yhat.values-actual7)):.0f} Wh/day — "
         "and crucially, storms now appear in the outlook 1-3 days ahead")

## 6. Layer A — Weekly Budget Optimizer (Linear Program)

**Decision variables:** `b[d]` = camera energy budget (Wh) for each of the next 7 days.

$$\max \sum_d b_d \quad\text{(footage ∝ energy spent)}$$

subject to, for every day-boundary $k$ (using **conservative** harvest $\hat H^{lo}_d$):

$$SoC_0 + \sum_{d\le k}\big(\eta\,\hat H^{lo}_d - hotel - b_d\big) \;\ge\; 240 \qquad \text{(floor, every day)}$$
$$SoC_0 + \sum_{d\le k}\big(\eta\,\hat H^{lo}_d - hotel - b_d\big) \;\le\; 1200 \qquad \text{(soft: avoid clipping)}$$
$$0 \le b_d \le 24\cdot 33.7$$

Why this produces *anticipatory* behavior: if day 4 is a forecast storm
($\hat H^{lo}_4 \approx 50$ Wh), the cumulative floor constraint at $k=4$ forces
$\sum_{d\le 4} b_d$ down — i.e. **days 1–3 automatically conserve** to bank charge.
No hand-written rules needed; it falls out of the LP geometry.

**⚠️ A receding-horizon pathology you must design around** (we hit this in testing):
if the objective is a plain $\max \sum_d b_d$, *any* distribution of the same total is
optimal, and the solver happily **back-loads** spending (b = [0, 0, …, big]). Under a
receding horizon that re-plans nightly, "spend later" becomes **"spend never"** — day 0's
budget is always 0, the battery pins at 100 %, and all surplus is clipped. Two fixes,
both applied below:

1. **Front-loaded objective weights** $w_d = 1 + 0.05(6-d)$ — ties are broken toward
   spending *today* (a Wh spent now is worth slightly more than a Wh promised for day 6).
2. **Clipping-avoidance constraints** — whenever cumulative inflow would exceed a full
   battery, force enough cumulative spend to absorb it:
   $\sum_{d\le k} b_d \ge SoC_0 + \sum_{d\le k} net_d - 1200$. These are dropped
   if they conflict with the safety floor (waste is acceptable; floor breach is not).

**Second pathology: floor-riding.** An LP will happily plan trajectories that touch the
floor *exactly* — then any forecast error breaches it. Defense in depth: a
`floor_margin_wh=70` planning buffer in the LP, a `floor_margin=60` execution buffer in
the hour selector, and bias-correction that is only allowed to *shrink* (never inflate)
the conservative harvest band used for planning.

In [ ]:
def solve_weekly_budget(soc0, harvest_lo_7, cfg=CFG, spend_end_frac=0.25,
                        floor_margin_wh=70.0, verbose=True):
    """LP over 7 daily budgets. Returns np.array(7) of Wh budgets.

    Design notes (see markdown above):
    • objective front-loads spending to defeat receding-horizon deferral
    • clipping constraints force spending when the battery would overflow
    • spend_end_frac keeps an end-of-horizon reserve (fraction of usable capacity)"""
    eta, hotel = cfg.charge_eff, cfg.hotel_w * 24
    net = eta * np.asarray(harvest_lo_7, float) - hotel   # net inflow before camera
    c = -(1.0 + 0.05*np.arange(6, -1, -1))                # front-loaded weights
    A_ub, b_ub = [], []
    for k in range(7):                                    # (a) floor at each boundary
        row = np.zeros(7); row[:k+1] = 1.0
        A_ub.append(row)
        floor = cfg.soc_floor_wh + floor_margin_wh \
                + (spend_end_frac*cfg.usable_wh if k == 6 else 0.0)
        b_ub.append(soc0 + net[:k+1].sum() - floor)
    clip_rows = []
    for k in range(7):                                    # (b) anti-clipping (soft)
        overflow = soc0 + net[:k+1].sum() - cfg.battery_wh
        if overflow > 0:
            row = np.zeros(7); row[:k+1] = -1.0           # -sum(b) <= -overflow
            clip_rows.append((row, -overflow))
    def _try(extra):
        A = np.array(A_ub + [r for r, _ in extra]); bb = np.array(b_ub + [v for _, v in extra])
        return linprog(c, A_ub=A, b_ub=bb, bounds=[(0, 24*cfg.camera_w)]*7, method="highs")
    res = _try(clip_rows)
    if not res.success:                                   # clip constraints conflict w/ floor
        res = _try([])
    if not res.success:                                   # even zero-camera breaches floor
        log.warning("Budget LP infeasible — full conservation mode (0 budgets)")
        return np.zeros(7)
    b = np.maximum(res.x, 0)
    if verbose:
        log.info("Weekly budget: " + " ".join(f"{x:5.0f}" for x in b) +
                 f"  Wh  (= {b.sum()/cfg.camera_w:.1f} camera-hours planned)")
    return b

# ---- demo: budget with vs without a forecast storm --------------------------
normal_wk = np.full(7, 210.0)
storm_wk  = np.array([210, 210, 210, 60, 50, 70, 210.])   # storm days 3-5
log.info("Scenario A — flat sunny week:")
bA = solve_weekly_budget(soc0=900, harvest_lo_7=normal_wk)
log.info("Scenario B — storm forecast on days 3-5:")
bB = solve_weekly_budget(soc0=900, harvest_lo_7=storm_wk)

x = np.arange(7); w=0.38
plt.figure(figsize=(9,3.5))
plt.bar(x-w/2, bA/CFG.camera_w, w, label="sunny week budget")
plt.bar(x+w/2, bB/CFG.camera_w, w, label="storm-days-3..5 budget")
plt.xlabel("day of plan"); plt.ylabel("camera-hours budgeted")
plt.title("Anticipatory conservation emerges from the LP —\ndays 0-2 pre-bank charge when a storm is forecast")
plt.legend(); plt.tight_layout(); plt.show()
log.info("Note days 0-2 in scenario B: budgets shrink *before* the storm arrives. "
         "This is the downward coupling in action.")

## 7. Layer B — Solar-Coincidence Hour Selector

Given **today's Wh budget** (Layer A) and the **LSTM's 24-h power forecast**, choose
which hours the camera runs.

**Scoring rule:** an hour's *battery cost* is `max(0, 33.7 − forecast_harvest[h])` —
i.e. how much of the camera draw the panel *cannot* cover. Greedy-pick cheapest hours
first (midday ≈ free; night = full 33.7 Wh from the battery).

Two safety passes after selection:
1. **SoC simulation** with the forecast — drop selected hours (most expensive first)
   until the simulated trajectory clears the floor with a margin.
2. **Runtime guard on the buoy** (documented in §10): hard low-SoC cutoff independent
   of any forecast — forecasting is *optimization*, the BMS is *safety*.

In [ ]:
def select_hours(budget_wh, fcst_24, soc0, cfg=CFG, floor_margin=60.0, verbose=False):
    """Greedy solar-coincident hour selection under an energy budget + SoC floor.
    Returns boolean schedule[24]."""
    battery_cost = np.maximum(0.0, cfg.camera_w - np.minimum(fcst_24, cfg.panel_w))
    order = np.argsort(battery_cost, kind="stable")        # cheapest hours first
    sched, spent = np.zeros(24, bool), 0.0
    for h in order:
        if spent + cfg.camera_w <= budget_wh:
            sched[h] = True; spent += cfg.camera_w
    # safety pass: forecast-simulated SoC must clear floor + margin
    for _ in range(24):
        socs, _, viol = simulate_day(soc0, fcst_24, sched, cfg)
        if socs.min() >= cfg.soc_floor_wh + floor_margin: break
        on = np.where(sched)[0]
        if len(on) == 0: break
        drop = on[np.argmax(battery_cost[on])]             # shed most expensive hour
        sched[drop] = False
        if verbose: log.info(f"  safety pass: dropped hour {drop:02d} (cost {battery_cost[drop]:.1f} Wh)")
    return sched

# ---- demo on one eval day ---------------------------------------------------
demo_fc = preds[0]; demo_true = trues[0]
sched = select_hours(budget_wh=6*CFG.camera_w, fcst_24=demo_fc, soc0=700)
socs, clip, viol = simulate_day(700, demo_true, sched)

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].plot(demo_true, label="actual solar", lw=2)
ax[0].plot(demo_fc, "--", label="LSTM forecast", lw=1.5)
ax[0].axhline(CFG.camera_w, color="r", ls=":", label="camera draw")
for h in np.where(sched)[0]: ax[0].axvspan(h-0.5, h+0.5, alpha=0.15, color="green")
ax[0].set(title=f"Hour selection (green = camera ON): {sched.sum()} h scheduled, "
                f"solar-coincident by construction", ylabel="W"); ax[0].legend()
ax[1].plot(socs, "k-o", ms=3, label="SoC trajectory (actual weather)")
ax[1].axhline(CFG.soc_floor_wh, color="r", ls="--", label="20% floor")
ax[1].axhline(CFG.battery_wh, color="g", ls=":", label="full")
ax[1].set(xlabel="hour", ylabel="Wh", title=f"Simulated SoC — floor violated: {viol}, clipped: {clip:.0f} Wh")
ax[1].legend(); plt.tight_layout(); plt.show()

## 8. The Joint Receding-Horizon Loop

Everything so far assembles into the nightly cycle. Per simulated day:

```
22:00  uplink telemetry → shore
       1. UPWARD: re-anchor SoC to *actual* end-of-day value
       2. UPWARD: bias-correct Prophet — ratio of (recent actual harvest)/(Prophet
          prediction for those days), clipped to [0.5, 1.5], multiplies remaining outlook
       3. Strategic: Prophet 7-day (lower band) → Budget LP → today's budget b[0]
       4. Tactical: LSTM 24-h forecast → hour selector → schedule[24]
       5. downlink 24 booleans to the buoy
06:00+ buoy EXECUTES against REAL weather (which differs from every forecast)
```

We run this closed loop over the full 30-day evaluation window, **through the forced
3-day frontal storm**, and log every decision.

In [ ]:
def run_closed_loop(full_df, daily_df, tactical, strategic, eval_start, n_days,
                    cfg=CFG, soc_init=0.75*1200, policy_name="hierarchical"):
    """The full Plan-1 controller. Returns per-day results DataFrame."""
    soc = soc_init
    rows = []
    bias = 1.0                                             # Prophet multiplicative correction
    recent_pairs = []                                      # (actual, prophet_pred) for bias calc
    for d in range(n_days):
        day_id = eval_start + d
        cut = full_df.timestamp.iloc[0] + pd.Timedelta(days=day_id)
        hist_h = full_df[full_df.timestamp < cut]
        hist_d = daily_df.iloc[:day_id]

        # --- strategic: 7-day outlook, bias-corrected, conservative band ------
        fc7 = strategic.predict_next_7d(hist_d)
        if recent_pairs:                                   # upward coupling #2
            a, p = np.array(recent_pairs[-7:]).T
            bias = float(np.clip((a.sum()+1e-9)/(p.sum()+1e-9), 0.5, 1.5))
        planning_bias = min(bias, 1.0)      # bias may only *reduce* the safe band
        harvest_lo = fc7.yhat_lower.values * planning_bias
        budgets = solve_weekly_budget(soc, harvest_lo, cfg, verbose=False)
        budget_today = budgets[0]

        # --- tactical: 24h forecast → hours -----------------------------------
        fc24 = tactical.predict_next_24h(hist_h)
        sched = select_hours(budget_today, fc24, soc, cfg)

        # --- execute against reality ------------------------------------------
        true24 = full_df[(full_df.timestamp >= cut) &
                         (full_df.timestamp < cut + pd.Timedelta(days=1))].solar_w.values
        socs, clip, viol = simulate_day(soc, true24, sched, cfg)
        soc_end = socs[-1]
        recent_pairs.append((true24.sum(), float(fc7.yhat.iloc[0])))

        rows.append(dict(day=d, regime=daily_df.regime.iloc[day_id],
                         budget_wh=budget_today, hours_on=int(sched.sum()),
                         footage_wh=sched.sum()*cfg.camera_w,
                         soc_start=soc, soc_end=soc_end, soc_min=socs.min(),
                         clipped_wh=clip, floor_violation=viol, bias=bias,
                         harvest_actual=true24.sum(), harvest_lo_fc=harvest_lo[0]))
        log.info(f"[{policy_name}] day {d:02d} {rows[-1]['regime']:<7s} | "
                 f"budget {budget_today:5.0f} Wh → {int(sched.sum()):2d} h ON | "
                 f"SoC {soc:5.0f}→{soc_end:5.0f} (min {socs.min():5.0f}) | "
                 f"bias {bias:.2f}{' | ⚠ FLOOR' if viol else ''}")
        soc = soc_end                                       # upward coupling #1
    return pd.DataFrame(rows)

log.info("="*80); log.info("CLOSED-LOOP SIMULATION — 30 eval days incl. 3-day frontal storm")
log.info("="*80)
res_hier = run_closed_loop(df, daily, tactical, strategic, N_TRAIN_DAYS, N_EVAL_DAYS)

In [ ]:
# ---- deep-dive visualization of the closed loop -----------------------------
fig, ax = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
storm_rel = [d - N_TRAIN_DAYS for d in STORM_DAYS]

ax[0].plot(res_hier.day, res_hier.soc_end, "k-o", ms=4, label="end-of-day SoC")
ax[0].plot(res_hier.day, res_hier.soc_min, "b:", label="intra-day min SoC")
ax[0].axhline(CFG.soc_floor_wh, color="r", ls="--", label="20% floor")
for s in storm_rel: ax[0].axvspan(s-0.5, s+0.5, color="red", alpha=0.08)
ax[0].set(ylabel="Wh", title="Battery trajectory — note pre-storm banking (SoC climbs before red band)")
ax[0].legend(loc="lower left")

ax[1].bar(res_hier.day, res_hier.hours_on, color="green", alpha=0.7, label="camera hours executed")
ax[1].plot(res_hier.day, res_hier.budget_wh/CFG.camera_w, "k--", lw=1, label="LP budget (hours)")
for s in storm_rel: ax[1].axvspan(s-0.5, s+0.5, color="red", alpha=0.08)
ax[1].set(ylabel="hours", title="Daily footage vs LP budget — conservation before/during storm, spend-up after")
ax[1].legend()

ax[2].bar(res_hier.day, res_hier.harvest_actual, alpha=0.5, label="actual harvest")
ax[2].plot(res_hier.day, res_hier.harvest_lo_fc, "r.-", lw=1, label="conservative forecast used by LP")
ax[2].plot(res_hier.day, res_hier.bias*200, "c:", label="Prophet bias corr ×200 (visual scale)")
for s in storm_rel: ax[2].axvspan(s-0.5, s+0.5, color="red", alpha=0.08)
ax[2].set(xlabel="evaluation day", ylabel="Wh", title="Strategic layer inputs — bias correction adapts within ~2 days")
ax[2].legend()
plt.tight_layout(); plt.show()

log.info(f"TOTALS — footage {res_hier.hours_on.sum()} h over {N_EVAL_DAYS} days "
         f"({res_hier.hours_on.mean():.1f} h/day) | floor violations: {res_hier.floor_violation.sum()} "
         f"| clipped {res_hier.clipped_wh.sum():.0f} Wh")

## 9. Baselines & Evaluation

Four policies, identical weather, identical starting SoC:

| Policy | Strategic layer | Tactical layer |
|---|---|---|
| **hierarchical** (ours) | Prophet-LP + bias-correction | LSTM + hour selector |
| **fixed-midday** | none | always ON 10:00–15:00 (5 h) |
| **tactical-only** | none — spends greedily every day | LSTM + selector, budget = yesterday's harvest |
| **strategic-only** | Prophet-LP | budget spread uniformly over daylight (no LSTM) |

**Scoring:** total footage · floor violations (must be 0) · clipping waste ·
footage captured during the storm window (resilience).

In [ ]:
def run_fixed_midday(full_df, daily_df, eval_start, n_days, cfg=CFG, soc_init=900):
    soc, rows = soc_init, []
    for d in range(n_days):
        cut = full_df.timestamp.iloc[0] + pd.Timedelta(days=eval_start+d)
        true24 = full_df[(full_df.timestamp>=cut)&(full_df.timestamp<cut+pd.Timedelta(days=1))].solar_w.values
        sched = np.zeros(24, bool); sched[10:15] = True
        # emulate BMS: if SoC below floor+margin at hour start, camera stays off
        s, socs, clip = soc, [soc], 0.0; viol=False
        for h in range(24):
            on = sched[h] and s > cfg.soc_floor_wh + cfg.camera_w
            s, c = step_soc(s, true24[h], on, cfg); clip += c
            viol |= s < cfg.soc_floor_wh; socs.append(s)
            sched[h] = on
        rows.append(dict(day=d, hours_on=int(sched.sum()), soc_end=s, soc_min=min(socs),
                         clipped_wh=clip, floor_violation=viol,
                         regime=daily_df.regime.iloc[eval_start+d]))
        soc = s
    return pd.DataFrame(rows)

class GreedyStrategic:
    """'tactical-only': budget = 90% of yesterday's net surplus. No 7-day lookahead."""
    name="greedy"
    def predict_next_7d(self, daily_hist):
        y = daily_hist.harvest_wh.iloc[-1]
        return pd.DataFrame({"yhat":[y]*7, "yhat_lower":[y]*7})

class UniformTactical:
    """'strategic-only': no hourly forecast at all — flat mean power across 24 h.
    The hour selector then has no solar-coincidence signal: hours are effectively
    arbitrary, so many land at night and burn battery. This isolates the LSTM's value."""
    name="flat-24h"
    def predict_next_24h(self, hist):
        mean_w = hist.solar_w.iloc[-72:].mean()
        return np.full(24, mean_w)

log.info("Running baseline: fixed-midday"); res_fixed = run_fixed_midday(df, daily, N_TRAIN_DAYS, N_EVAL_DAYS)
log.info("Running baseline: tactical-only (no weekly outlook)")
res_tact = run_closed_loop(df, daily, tactical, GreedyStrategic(), N_TRAIN_DAYS, N_EVAL_DAYS,
                           policy_name="tactical-only")
log.info("Running baseline: strategic-only (no LSTM)")
res_strat = run_closed_loop(df, daily, UniformTactical(), strategic, N_TRAIN_DAYS, N_EVAL_DAYS,
                            policy_name="strategic-only")

In [ ]:
def score(name, r):
    storm_mask = r.day.isin([d - N_TRAIN_DAYS for d in STORM_DAYS]) if "day" in r else None
    return dict(policy=name,
                footage_h=int(r.hours_on.sum()),
                h_per_day=round(r.hours_on.mean(),2),
                floor_violations=int(r.floor_violation.sum()),
                clipped_Wh=int(r.clipped_wh.sum()),
                storm_footage_h=int(r[r.regime=="FRONTAL"].hours_on.sum()),
                min_soc=int(r.soc_min.min()))

scores = pd.DataFrame([score("hierarchical (ours)", res_hier),
                       score("fixed-midday", res_fixed),
                       score("tactical-only", res_tact),
                       score("strategic-only", res_strat)]).set_index("policy")
print(scores.to_string())

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
scores.footage_h.plot.bar(ax=ax[0], color=["#2ca02c","#999","#1f77b4","#ff7f0e"], rot=20)
ax[0].set_title("Total footage hours ↑")
scores.floor_violations.plot.bar(ax=ax[1], color=["#2ca02c","#999","#1f77b4","#ff7f0e"], rot=20)
ax[1].set_title("Floor violations ↓ (safety — must be 0)")
scores.clipped_Wh.plot.bar(ax=ax[2], color=["#2ca02c","#999","#1f77b4","#ff7f0e"], rot=20)
ax[2].set_title("Clipped (wasted) energy ↓")
plt.tight_layout(); plt.show()

log.info("READING THE RESULTS:")
log.info(" • hierarchical should match/beat others on footage WITH zero violations")
log.info(" • tactical-only typically violates the floor (or starves) in the storm —")
log.info("   it cannot see the storm coming, so it doesn't pre-bank charge")
log.info(" • fixed-midday is safe but wastes clipping surplus & under-spends sunny weeks")
log.info(" • strategic-only wastes battery running non-solar-coincident hours")

## 10. Uncertainty Handling & Deployment Notes

### Risk posture knobs (all exposed above)
| Knob | Where | Effect |
|---|---|---|
| `yhat_lower` vs `yhat` | §6 LP input | plan pessimistic (safe) vs expected (more footage) |
| `spend_end_frac` | `solve_weekly_budget` | end-of-horizon reserve — prevents myopic day-7 drain |
| `floor_margin` | `select_hours` | intra-day buffer above the 240 Wh floor |
| bias-correction clip `[0.5, 1.5]` | §8 loop | limits how fast one weird day skews the outlook |

### Deployment sketch
* **Shore-side nightly cron** (22:00 CT): pull telemetry → retrain/refresh LSTM weekly,
  refit Prophet nightly (seconds) → LP → downlink 24-boolean schedule (+CRC).
* **On-buoy fallback** if no downlink by 23:30: static conservative schedule
  (3 h centered on solar noon) *and* the BMS hard cutoff — the forecast stack is an
  optimizer, never the safety layer.
* **Intra-day re-plan trigger** (optional): if by 10:00 the morning's actual harvest is
  < 60 % of the LSTM forecast, re-run selector with remaining budget × 0.7.
* **Monitoring**: log `bias` drift (persistent > 1.3 → Prophet stale, retrain),
  LSTM walk-forward MAE (alert > 2× training MAE → concept drift, e.g. panel biofouling).

### What Plan 2 changes
Plan 1 needs two model codebases, two retraining loops, and a hand-built bias bridge.
The companion notebook (**Plan 2**) collapses LSTM + Prophet into a single TimeGPT-1
API with calibrated intervals — same LP + hour selector, radically less ML plumbing,
plus anomaly detection (biofouling/soiling) for free.
